In [68]:
import os
import sys
import io
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pandas as pd
import numpy as np
from tqdm import tqdm
import cv2
from PIL import Image
import torch
import gc 
import warnings

source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

warnings.filterwarnings('ignore')

In [82]:
random_tensor = torch.randn(3, 224, 224)
test_image=generate_cross_image(cross_width_ratio=0.5, cross_height_ratio=0.5)
model_list=global_vars.list_models()
N_max=282
patches=True
pooling=False # if true in transformer mdoels use pooling, if false only the cls token
custom_transform=False
transform_mode='resize'
save_h5=False
truncation = 'remove head'
running = 'new-laptop'
saved = 'old-laptop'
model_mode = 'truncated' #'as is', 'truncated
batch_size = 32
select_cls=False
num_workers=0
normalization=True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device is: ",device)

Device is:  cuda


In [83]:
df_pre_patch, df_pre_extracted,_,_ =file_IO.preprocess_experiment_logs(source_path)

In [84]:
squares_standard_name = 'squares_gw5.0_m5.0_idx2' #'rectangles_gw3.0_m5.0_idx1' #'squares_gw5.0_m5.0_idx2'
print(file_IO.get_experiment_from_unique_name(df_pre_patch, squares_standard_name))
input_file_name='icdar_train_df_patches_20250515_164130.csv'
viz_numeric=file_IO.get_reference_table_for_experiments(df_pre_patch,df_pre_extracted,im_show=False,im_plot=False)
already_used_models=file_IO.get_models_applied_to_unique_name(viz_numeric, squares_standard_name)
print("Already used models on ",squares_standard_name,": ",already_used_models)
already_used_models=[] # to re-run all models

icdar_train_df_patches_20250515_164130.csv
Already used models on  squares_gw5.0_m5.0_idx2 :  ['trocr-small-stage1', 'trocr-small-handwritten', 'resnet50', 'vit-base-patch16-224-in21k', 'trocr-base-handwritten', 'clip-vit-large-patch14', 'trocr-large-handwritten', 'dresnet50', 'crnn_vgg16_bn', 'vitstr_base', 'trocr-large-stage1', 'trocr-base-stage1', 'alexnet', 'vgg16', 'googlenet', 'resnet18', 'DeiT-Tiny', 'swin_b', 'swin_s', 'DeiT-Small', 'DeiT-Small-Dist', 'DeiT-Base', 'BEiT-Base', 'clip-vit-base-patch16', 'clip-vit-base-patch32', 'clip-vit-large-patch14-un', 'vit-base-patch16-224', 'vit-base-patch32-224-in21k', 'vit-large-patch16-224-in21k', 'vit-huge-patch14-224-in21k', 'vitstr_small', 'db_mobilenet', 'crnn_mobilenet_224', 'linknet_resnet50_224', 'linknet_resnet18', 'crnn_mobilenet', 'sar_resnet31', 'alexnet_gap', 'densenet161', 'densenet121', 'densenet201', 'vgg11', 'vgg16_512', 'maxvit', 'efficientnet_v2_l', 'efficientnet_v2_s', 'convnext_large', 'convnext_base', 'convnext_small

In [85]:
from utils import global_vars


def add_to_df(name,df_models,props):
    df_models.append({'FE model':name})
    df_models[-1]['pretraining_mode']=props.pretraining_mode
    df_models[-1]['pretraining_dataset']=props.pretraining_dataset
    df_models[-1]['architecture_specific']=props.architecture_specific
    df_models[-1]['architecture']=props.architecture
    df_models[-1]['library']=props.library
    return df_models
df_models=[]
for i in range(0,min(len(model_list),len(model_list))):
    name=model_list[i]
    print(i)
    if name in already_used_models: 
        '''print("Skipping already used model: ",name)
        print('-'*40)
        print('-'*40)
        continue'''
        print("Already used")
    print("Using model: ", name)
    huggingface=global_vars.get_props(name).hugging
    selected_model=name
    transform = u_transforms.get_transform(selected_model, use_patches=patches, custom=custom_transform, mode=transform_mode)
    model = model_utils.get_model(name=selected_model, mode=model_mode, pretrained=True, truncation=truncation)
    torch.cuda.empty_cache()
    gc.collect() 
    model = model.to(device)
    output,input_size,processed_img=compute_output(model, device, transform, huggingface,test_image,show_image=False)
    input_size=list(input_size[1:])  # remove batch dim
    if output.shape[0]==1:
        output_shape = output.shape[1:]
    else:
        output_shape = output.shape[0]
    output_dim = int(output_shape[0])
    print("Output shape: ", output_dim)
    #n_par,n_layers = count_model_parameters_and_layers(model)
    #print('n parameters',f"{n_par:.3e}")
    #print('n layers',n_layers)
    n_par_tot=count_params(model, trainable_only=False, include_buffers=False)
    n_par=used_params_via_hooks(model, x=processed_img.to(device))
    depth=forward_depth(model, x=processed_img.to(device))
    print('n parameters total',f"{n_par_tot:.3e}")
    print('n parameters used',f"{n_par:.3e}")
    print('forward depth',depth)
    #print(transform)
    print('-'*40)
    print('-'*40)
    props=global_vars.get_props(name)
    df_models=add_to_df(name,df_models,props)
    df_models[-1]['n_parameters_tot']=n_par_tot
    df_models[-1]['n_parameters_used']=n_par
    df_models[-1]['forward_depth']=depth
    df_models[-1]['output_shape']=output_dim
    df_models[-1]['input_size']=input_size
    #print(df_models[name])

0
Using model:  swin_b
Output shape:  1024
n parameters total 8.674e+07
n parameters used 5.878e+07
forward depth 287
----------------------------------------
----------------------------------------
1
Using model:  swin_s
Output shape:  768
n parameters total 4.884e+07
n parameters used 3.310e+07
forward depth 287
----------------------------------------
----------------------------------------
2
Using model:  BEiT-Base
Output shape:  768
n parameters total 8.674e+07
n parameters used 8.674e+07
forward depth 251
----------------------------------------
----------------------------------------
3
Using model:  BEiT-Large
Output shape:  1024
n parameters total 3.050e+08
n parameters used 3.050e+08
forward depth 491
----------------------------------------
----------------------------------------
4
Using model:  BEiT-Large-inter
Output shape:  1024
n parameters total 3.040e+08
n parameters used 3.040e+08
forward depth 465
----------------------------------------
--------------------------

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Output shape:  197
n parameters total 8.639e+07
n parameters used 8.639e+07
forward depth 214
----------------------------------------
----------------------------------------
6
Using model:  vit-base-patch32-224-in21k
Output shape:  50
n parameters total 8.805e+07
n parameters used 8.805e+07
forward depth 214
----------------------------------------
----------------------------------------
7
Using model:  vit-large-patch16-224-in21k
Output shape:  197
n parameters total 3.044e+08
n parameters used 3.044e+08
forward depth 418
----------------------------------------
----------------------------------------
8
Using model:  vit-huge-patch14-224-in21k
Output shape:  257
n parameters total 6.324e+08
n parameters used 6.324e+08
forward depth 554
----------------------------------------
----------------------------------------
9
Using model:  vit-base-patch16-224-in21k
Output shape:  197
n parameters total 8.639e+07
n parameters used 8.639e+07
forward depth 214
------------------------------

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-large-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Output shape:  577
n parameters total 3.047e+08
n parameters used 3.047e+08
forward depth 418
----------------------------------------
----------------------------------------
33
Using model:  trocr-large-handwritten-inter


...

Output shape:  577
n parameters total 8.665e+07
n parameters used 8.665e+07
forward depth 214
----------------------------------------
----------------------------------------
38
Using model:  trocr-small-handwritten


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-small-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Output shape:  578
n parameters total 2.196e+07
n parameters used 2.196e+07
forward depth 214
----------------------------------------
----------------------------------------
39
Using model:  trocr-small-stage1


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-small-stage1 and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Output shape:  578
n parameters total 2.196e+07
n parameters used 2.196e+07
forward depth 214
----------------------------------------
----------------------------------------
40
Using model:  dresnet50
Output shape:  2048
n parameters total 2.351e+07
n parameters used 2.351e+07
forward depth 150
----------------------------------------
----------------------------------------
41
Using model:  dresnet50-inter
Output shape:  512
n parameters total 2.351e+07
n parameters used 2.351e+07
forward depth 150
----------------------------------------
----------------------------------------
42
Using model:  db_mobilenet
Output shape:  960
n parameters total 2.972e+06
n parameters used 2.972e+06
forward depth 248
----------------------------------------
----------------------------------------
43
Using model:  linknet_resnet50_224
Output shape:  2048
n parameters total 2.351e+07
n parameters used 2.351e+07
forward depth 149
----------------------------------------
-------------------------------

In [ ]:
print(global_vars.get_props("swin_v2_b"))
print(global_vars.get_props("swin_v2_b").hugging)
print(global_vars.list_models()[:10])

ModelProps(hugging=False, library='torchvision', architecture='transformer')
False
['swin_v2_b', 'swin_v2_s', 'DeiT-Small', 'DeiT-Small-Dist', 'DeiT-Base', 'BEiT-Base', 'BEiT-Large', 'clip-vit-base-patch16', 'clip-vit-base-patch32', 'clip-vit-large-patch14-un']


# join with result df

## join

In [86]:
df_this = pd.DataFrame(df_models)
df_loaded = pd.read_csv(r"C:\\Users\\andre\\VsCode\\PD related projects\\gender_detection\\notebooks\\model_comparison_experiment\\results\\df_aggregated_last.csv")
df_merged = pd.merge(df_this, df_loaded, on='FE model', how='inner')

In [87]:
display(df_merged.head())
print(df_merged.columns)

,FE model,pretraining_mode,pretraining_dataset,architecture_specific,architecture,library,n_parameters_tot,n_parameters_used,forward_depth,output_shape,...,ind_accuracies_single,ind_accuracies_std_single,accuracies_body,accuracies_std_body,ind_accuracies_body,ind_accuracies_std_body,generalization_standard,generalization_std_standard,generalization_ind_standard,generalization_ind_std_standard
0,swin_b,classification,imagenet1k,swin-b,transformer,torchvision,86743224,58776760,287,1024,...,0.676108,0.077101,0.689532,0.023798,0.689532,0.023798,0.068288,0.079038,0.047432,0.064667
1,swin_s,classification,imagenet1k,swin-s,transformer,torchvision,48837258,33097098,287,768,...,0.687716,0.055332,0.661915,0.057711,0.661915,0.057711,0.121275,0.115644,0.072143,0.084095
2,BEiT-Base,masking,imagenet21k,beit-b,transformer,huggingface,86744104,86744104,251,768,...,0.677371,0.038426,0.664840,0.052301,0.664840,0.052301,0.133005,0.122325,0.083134,0.088556
3,BEiT-Large,masking,imagenet21k,beit-l,transformer,huggingface,304998888,304998888,491,1024,...,0.656989,0.025901,0.657635,0.055780,0.657635,0.055780,0.151724,0.101445,0.100979,0.076872
4,BEiT-Large-inter,masking,imagenet21k,beit-l,transformer,huggingface,303973888,303973888,465,1024,...,0.717180,0.049545,0.731927,0.044698,0.731927,0.044698,0.133621,0.076265,0.095536,0.065845


Index(['FE model', 'pretraining_mode', 'pretraining_dataset',
       'architecture_specific', 'architecture', 'library', 'n_parameters_tot',
       'n_parameters_used', 'forward_depth', 'output_shape', 'input_size',
       'accuracies_standard', 'accuracies_std_standard',
       'ind_accuracies_standard', 'ind_accuracies_std_standard',
       'accuracies_single', 'accuracies_std_single', 'ind_accuracies_single',
       'ind_accuracies_std_single', 'accuracies_body', 'accuracies_std_body',
       'ind_accuracies_body', 'ind_accuracies_std_body',
       'generalization_standard', 'generalization_std_standard',
       'generalization_ind_standard', 'generalization_ind_std_standard'],
      dtype='object')


In [88]:
test=df_merged[df_merged['accuracies_standard']>=0.7][['FE model','accuracies_standard','input_size']]
print(len(test))
display(test)

32


,FE model,accuracies_standard,input_size
0,swin_b,0.723337,"[256, 256]"
1,swin_s,0.729526,"[256, 256]"
2,BEiT-Base,0.765025,"[384, 384]"
3,BEiT-Large,0.767826,"[384, 384]"
4,BEiT-Large-inter,0.766041,"[384, 384]"
5,vit-base-patch16-224,0.721829,"[224, 224]"
8,vit-huge-patch14-224-in21k,0.706158,"[224, 224]"
9,vit-base-patch16-224-in21k,0.712685,"[224, 224]"
10,DeiT-Small,0.730357,"[224, 224]"
11,DeiT-Small-Dist,0.713300,"[224, 224]"


In [89]:
df_merged['architecture_specific'].value_counts()

architecture_specific
beit-l                  6
resnet50                5
resnet34                4
vgg16                   4
mobilenet_v3_l          4
vit-l/14                3
vit-s                   3
deit-s                  3
vit-b/16                3
beit-b                  3
vit-b                   3
alexnet                 2
resnet31                2
vit-t                   2
vit-b/32                2
resnet18                2
swin-b                  1
swin-s                  1
vit-l/16                1
vit-h/14                1
densenet161             1
googlenet               1
densenet201             1
vgg11                   1
vgg16_512               1
densenet121             1
maxvit                  1
inception_net           1
efficientnet_v2_s       1
efficientnet_v2_l       1
convnext_large-inter    1
convnext_base           1
convnext_small          1
convnext_large          1
mobilenet_v3_small      1
mobilenet_v3_large      1
resnet101               1
regnet_x_32     

In [90]:
df_merged[df_merged['architecture_specific']=='beit-l'][['accuracies_standard','accuracies_std_standard','FE model','pretraining_dataset']].sort_values(by='accuracies_standard', ascending=False)

,accuracies_standard,accuracies_std_standard,FE model,pretraining_dataset
3,0.767826,0.061211,BEiT-Large,imagenet21k
4,0.766041,0.051911,BEiT-Large-inter,imagenet21k
33,0.752555,0.052081,trocr-large-handwritten-inter,trocr+iam
35,0.734144,0.070820,trocr-large-stage1-inter,trocr
34,0.724969,0.064788,trocr-large-stage1,trocr
32,0.711700,0.048113,trocr-large-handwritten,trocr+iam


## Best per class (paper table)

In [106]:
df=df_merged.copy()
grouping_cols=['architecture','pretraining_mode']
cols_of_interest=grouping_cols+['models_in_class','largest_in_class']+['FE model','accuracies_standard','ind_accuracies_standard','accuracies_single','accuracies_body',
                  'generalization_standard','n_parameters_used','forward_depth','output_shape']
model_names=df['FE model'].tolist()
with_inter=[name.replace('-inter','') for name in model_names if '-inter' in name]
df['inter']=df['FE model'].apply(lambda x: True if x in with_inter or '-inter' in x else False)
df['models_in_class']=0
df['largest_in_class']=''

In [107]:
df['models_in_class'] = df.groupby(grouping_cols)['FE model'].transform('count')
df['largest_in_class']= df.groupby(grouping_cols)['n_parameters_used'].transform('idxmax').apply(lambda x: df.loc[x,'FE model'])
df_grouped = df.groupby(grouping_cols+['inter'])[cols_of_interest].apply(lambda x: x.sort_values(by='accuracies_standard', ascending=False).head(1)).reset_index(drop=True)
#display(df_grouped.sort_values(by='accuracies_standard', ascending=False))
df['false_accuracy']=df['accuracies_standard']
included_models=df_grouped['FE model'].tolist()
to_add=[]
for name in included_models:
    if 'inter' not in name:
        if name in with_inter:
            to_add.append(name+'-inter')
    else:
        to_add.append(name.replace('-inter',''))

included_models+=to_add
df_final = df[df['FE model'].isin(included_models)]
df_final['false_accuracy'] = df_final.groupby(grouping_cols)['accuracies_standard'].transform('max')
print(len(df_final))
display(df_final.sort_values(by=['false_accuracy','accuracies_standard'], ascending=False)[['inter']+cols_of_interest])


23


,inter,architecture,pretraining_mode,models_in_class,largest_in_class,FE model,accuracies_standard,ind_accuracies_standard,accuracies_single,accuracies_body,generalization_standard,n_parameters_used,forward_depth,output_shape
19,True,transformer,contrastive,5,clip-vit-large-patch14-un,clip-vit-large-patch14-inter,0.774015,0.740062,0.735099,0.723430,0.092149,303179776,295,1024
17,False,transformer,contrastive,5,clip-vit-large-patch14-un,clip-vit-large-patch14-un,0.741010,0.685979,0.649507,0.648984,0.149723,303966208,296,768
18,True,transformer,contrastive,5,clip-vit-large-patch14-un,clip-vit-large-patch14,0.723307,0.692605,0.666841,0.649723,0.194089,303966208,296,768
3,True,transformer,masking,3,BEiT-Large,BEiT-Large,0.767826,0.718713,0.656989,0.657635,0.151724,304998888,491,1024
4,True,transformer,masking,3,BEiT-Large,BEiT-Large-inter,0.766041,0.729649,0.717180,0.731927,0.133621,303973888,465,1024
2,False,transformer,masking,3,BEiT-Large,BEiT-Base,0.765025,0.711188,0.677371,0.664840,0.133005,86744104,251,768
33,True,transformer,text_recognition,12,trocr-large-handwritten,trocr-large-handwritten-inter,0.752555,0.693836,0.669089,0.671644,0.128664,304666624,418,1024
32,True,transformer,text_recognition,12,trocr-large-handwritten,trocr-large-handwritten,0.711700,0.663510,0.617611,0.643381,0.082204,304666624,418,577
37,False,transformer,text_recognition,12,trocr-large-handwritten,trocr-base-stage1,0.710037,0.678584,0.662223,0.617642,0.109821,86653440,214,577
61,True,cnn,classification,30,regnet_y_128,convnext_large,0.734883,0.698615,0.677094,0.689409,0.100185,196227264,380,1536


## model names and properties table

In [115]:
df=df_merged.copy()
grouping_cols=['architecture','pretraining_mode']
df_unique = df.groupby(grouping_cols).first().reset_index()
display(df_unique)

,architecture,pretraining_mode,FE model,pretraining_dataset,architecture_specific,library,n_parameters_tot,n_parameters_used,forward_depth,output_shape,...,ind_accuracies_single,ind_accuracies_std_single,accuracies_body,accuracies_std_body,ind_accuracies_body,ind_accuracies_std_body,generalization_standard,generalization_std_standard,generalization_ind_standard,generalization_ind_std_standard
0,cnn,classification,resnet50,imagenet1k,resnet50,torchvision,23508032,23508032,150,2048,...,0.616749,0.051361,0.637808,0.046869,0.637808,0.046869,0.116995,0.099948,0.074557,0.077104
1,cnn,text_detection,dresnet50,doctr,resnet50,doctr,23508032,23508032,150,2048,...,0.650216,0.065795,0.684791,0.055032,0.684791,0.055032,0.191379,0.077655,0.137260,0.061335
2,cnn,text_recognition,crnn_mobilenet,doctr,mobilenet_v3_l,doctr,2971952,2971952,249,960,...,0.573615,0.037516,0.583097,0.028666,0.583097,0.028666,0.038793,0.094547,0.029243,0.066310
3,hybrid,classification,maxvit,imagenet1k,maxvit,torchvision,30143944,30143944,660,512,...,0.639778,0.048617,0.618473,0.045089,0.618473,0.045089,0.139224,0.097896,0.092389,0.061362
4,transformer,classification,swin_b,imagenet1k,swin-b,torchvision,86743224,58776760,287,1024,...,0.676108,0.077101,0.689532,0.023798,0.689532,0.023798,0.068288,0.079038,0.047432,0.064667
5,transformer,classification+distillation,DeiT-Small,imagenet1k,vit-s,huggingface,22050664,22050664,213,384,...,0.670105,0.068560,0.669212,0.055210,0.669212,0.055210,0.091318,0.087471,0.052118,0.052475
6,transformer,contrastive,clip-vit-base-patch16,Clip-vit,vit-b/16,huggingface,149620737,86192640,152,512,...,0.639039,0.079754,0.652986,0.043341,0.652986,0.043341,0.182328,0.110882,0.145954,0.078400
7,transformer,masking,BEiT-Base,imagenet21k,beit-b,huggingface,86744104,86744104,251,768,...,0.677371,0.038426,0.664840,0.052301,0.664840,0.052301,0.133005,0.122325,0.083134,0.088556
8,transformer,text_recognition,vitstr_base,doctr,vit-b,doctr,85196544,85196544,129,768,...,0.548892,0.035429,0.536238,0.036906,0.536238,0.036906,0.024507,0.091421,0.016773,0.052100


## Q5: inter vs non-inter

In [69]:
cols_of_interest=['FE model','inter','accuracies_standard','accuracies_std_standard']
df_compare=df_merged.copy()
model_names=df_merged['FE model'].tolist()
with_inter=df_compare[df_compare['FE model'].str.contains('inter')]['FE model'].tolist()
stemmed_names=[]
for name in with_inter:
    stemmed_names.append(name.replace('-inter',''))
all_names=stemmed_names + with_inter
df_compare['inter'] = df_compare['FE model'].isin(with_inter)
df_compare.loc[df_compare['FE model'].str.endswith('-inter'), 'FE model'] = (
    df_compare['FE model'].str.replace('-inter', '', regex=False)
)
selected=df_compare[df_compare['FE model'].isin(all_names)][cols_of_interest]#.sort_values(by='FE model', ascending=False))
display(selected.sort_values(by=['FE model','inter'], ascending=False))
diffs = selected.groupby('FE model').apply(lambda g: g.sort_values(by='inter').iloc[0]['accuracies_standard'] - g.sort_values(by='inter').iloc[1]['accuracies_standard'] if len(g) > 1 else np.nan)
print(diffs)

,FE model,inter,accuracies_standard,accuracies_std_standard
35,trocr-large-stage1,True,0.734144,0.070820
34,trocr-large-stage1,False,0.724969,0.064788
33,trocr-large-handwritten,True,0.752555,0.052081
32,trocr-large-handwritten,False,0.711700,0.048113
41,dresnet50,True,0.709052,0.049691
40,dresnet50,False,0.715148,0.050710
27,crnn_vgg16_bn_224,True,0.704495,0.068534
26,crnn_vgg16_bn_224,False,0.679526,0.093259
22,crnn_mobilenet_224,True,0.670751,0.056131
21,crnn_mobilenet_224,False,0.644243,0.059490


FE model
BEiT-Large                 0.001786
DeiT-Tiny                 -0.006342
clip-vit-large-patch14    -0.050708
convnext_large             0.007081
crnn_mobilenet_224        -0.026509
crnn_vgg16_bn_224         -0.024969
dresnet50                  0.006096
trocr-large-handwritten   -0.040856
trocr-large-stage1        -0.009175
dtype: float64


# reload

In [81]:
import matplotlib.pyplot as plt
def reload_modules():
    import importlib
    import utils.model_utils as model_utils
    import utils.utils_transforms as u_transforms
    import utils.global_vars as global_vars
    import utils.file_IO as file_IO

    importlib.reload(model_utils)
    importlib.reload(u_transforms)
    importlib.reload(global_vars)
    importlib.reload(file_IO)

    return model_utils, u_transforms, global_vars,file_IO
model_utils, u_transforms, global_vars, file_IO = reload_modules()
def generate_cross_image(resolution=(512, 512), cross_width_ratio=0.1, cross_height_ratio=0.1, color=(0, 0, 0)):
    """
    Generates an RGB image with a cross at its center.

    Args:
        resolution (tuple): The resolution of the image (width, height).
        cross_width_ratio (float): The ratio of the cross width to the image width.
        cross_height_ratio (float): The ratio of the cross height to the image height.
        color (tuple): The color of the cross in RGB format (0-255).

    Returns:
        np.ndarray: The generated image as a NumPy array.
    """
    width, height = resolution
    image = np.zeros((height, width, 3), dtype=np.uint8)+255

    cross_width = int(width * cross_width_ratio)
    cross_height = int(height * cross_height_ratio)

    # Draw horizontal bar of the cross
    start_x = (width - cross_width) // 2
    end_x = start_x + cross_width
    start_y_h = (height - int(height * 0.05)) // 2 # Small height for the horizontal bar
    end_y_h = start_y_h + int(height * 0.05)
    cv2.rectangle(image, (start_x, start_y_h), (end_x, end_y_h), color, -1)

    # Draw vertical bar of the cross
    start_x_v = (width - int(width * 0.05)) // 2 # Small width for the vertical bar
    end_x_v = start_x_v + int(width * 0.05)
    start_y = (height - cross_height) // 2
    end_y = start_y + cross_height
    cv2.rectangle(image, (start_x_v, start_y), (end_x_v, end_y), color, -1)

    return image
def compute_output(model, device, transform, huggingface,test_image, show_image=True):
    model.eval()
    if isinstance(test_image, np.ndarray):
      patch=Image.fromarray(test_image)
    else:
      patch = test_image
    if huggingface:
        # the transform is actually an huggingface processor in this case
        inputs = transform(images=patch, return_tensors="pt")
        # Remove batch dimension from inputs
        patch = inputs['pixel_values'].squeeze()
    else:
        patch = transform(patch)
    image_size=patch.shape  # (H, W)

    if show_image:
      img_np = patch.permute(1, 2, 0).cpu().numpy()
      # Normalize if needed
      print("min,max",img_np.min(),img_np.max())
      img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

      plt.subplot(1, 2, 2)
      plt.imshow(img_np)
      plt.title("Transformed")
      plt.axis('off')
      plt.show()

    patch = patch.to(device)
    output = model(patch.unsqueeze(0))
    return output,image_size, patch.unsqueeze(0).cpu()
# Example usage:
# image = generate_cross_image()
# Image.fromarray(image).save("cross_image.png")
def count_model_parameters_and_layers(model):
    """
    Counts the number of parameters and layers in a PyTorch model.

    Args:
        model (torch.nn.Module): The PyTorch model.

    Returns:
        tuple: A tuple containing the total number of parameters and the number of layers.
    """
    num_parameters = sum(p.numel() for p in model.parameters())
    num_layers = len(list(model.children()))
    return num_parameters, num_layers

def forward_depth(model, x):
    visited = set()
    handles = []

    def hook(m, inp, out):
        visited.add(m)

    for m in model.modules():
        handles.append(m.register_forward_hook(hook))

    model.eval()
    with torch.no_grad():
        _ = model(x)

    for h in handles: h.remove()
    return len(visited) - 1  # minus the root

def used_params_via_hooks(model, x, include_buffers=False):
    visited = set()
    handles = []

    def hook(m, inp, out):
        visited.add(m)

    for m in model.modules():
        handles.append(m.register_forward_hook(hook))

    # run once with your representative input
    model.eval()
    with torch.no_grad():
        _ = model(x)

    for h in handles: h.remove()

    # count unique tensors to avoid double-counting shared/tied weights
    seen_ptrs = set()
    def add_unique(t):
        if t is None: return 0
        ptr = t.data_ptr()
        if ptr in seen_ptrs: return 0
        seen_ptrs.add(ptr)
        return t.numel()

    used = 0
    for m in visited:
        for p in m.parameters(recurse=False):
            used += add_unique(p.data)
        if include_buffers:
            for b in m.buffers(recurse=False):
                used += add_unique(b.data)
    return used

def count_params(model, trainable_only=False, include_buffers=False):
    params = (p for p in model.parameters() if (p.requires_grad or not trainable_only))
    n_params = sum(p.numel() for p in params)
    if include_buffers:
        n_params += sum(b.numel() for b in model.buffers())
    return n_params

